# 语义分割数据集配置

## 1. 锁定工作目录

每次运行前，确保系统定位在 `mmsegmentation` 的根目录下。

In [1]:
import os

WORK_DIR = '/root/LearningMMSegmentation/mmsegmentation'
if os.path.exists(WORK_DIR):
    os.chdir(WORK_DIR)
    print(f"✅ 成功进入工作目录: {os.getcwd()}")
else:
    print(f"❌ 找不到路径 {WORK_DIR}，请检查文件夹。")
    
# 创建存放自己配置文件的专属文件夹
CONFIG_DIR = 'My-Configs'
os.makedirs(CONFIG_DIR, exist_ok=True)

✅ 成功进入工作目录: /root/LearningMMSegmentation/mmsegmentation


## 2. 定义数据集元信息 (Meta Info)

**💡 核心逻辑**：MMSegmentation 必须通过一个专门的类（Class）来认识你的数据集。
**🔧 换数据集，只需要修改下面代码框里的 `classes` 和 `palette`！**

*使用 Jupyter 的 `%%writefile` 命令，直接把下面的代码写成 `mmseg/datasets/CustomDataset.py` 文件。*

In [2]:
%%writefile mmseg/datasets/CustomDataset.py
# 👆 这行魔法命令会直接把当前格子的内容保存为 Python 文件

from mmseg.registry import DATASETS
from .basesegdataset import BaseSegDataset

@DATASETS.register_module()
class CustomDataset(BaseSegDataset):
    """
    你的自定义数据集类。
    """
    
    # ========================================================
    # 🔴 以后换数据集，只修改这下面的 classes 和 palette！
    # ========================================================
    METAINFO = dict(
        # 1. 类别名称 (务必与你的真实标注对应，注意英文逗号)
        classes=(
            'red_flesh', 
            'green_rind', 
            'white_rind', 
            'black_seed', 
            'white_seed', 
            'unlabeled'
        ),
        
        # 2. 类别对应的可视化颜色 (B, G, R 格式)
        palette=[
            [50, 50, 255],   # red_flesh (红色)
            [71, 255, 10],   # green_rind (绿色)
            [204, 204, 255], # white_rind
            [0, 0, 0],       # black_seed
            [119, 224, 210], # white_seed
            [150, 100, 200]  # unlabeled
        ]
    )
    # ========================================================

    def __init__(self, **kwargs):
        super().__init__(img_suffix='.jpg', seg_map_suffix='.png', **kwargs)

Overwriting mmseg/datasets/CustomDataset.py


## 3. 将新类目注入框架的 `__init__.py`

通过“2”写好了 `CustomDataset.py`，必须在框架的“目录”里登记一下，框架才能调用它。
“3.”会自动把类名追加到 `mmseg/datasets/__init__.py` 中，**以后都不用再手动去改那个文件了！**

In [3]:
init_file = 'mmseg/datasets/__init__.py'

# 读取原有的 init 文件内容
with open(init_file, 'r') as f:
    content = f.read()

# 检查是否已经注册过，避免重复写入
if 'CustomDataset' not in content:
    with open(init_file, 'a') as f:
        f.write("\n# 自定义数据集注册\n")
        f.write("from .CustomDataset import CustomDataset\n")
        f.write("__all__.append('CustomDataset')\n")
    print("✅ 成功将 CustomDataset 注册到系统中！")
else:
    print("⚡ CustomDataset 已存在于系统中，无需重复注册。")

⚡ CustomDataset 已存在于系统中，无需重复注册。


## 4. 构建 Pipeline 与 DataLoader 配置

这里是深度学习工程最核心的区域。为了清楚知道**哪里必须改**和**哪里可以用来调优**，将变量严格分成了三个区域。

In [4]:
import os
from mmengine import ConfigDict

### 📍 区域一：路径配置区 (换数据集时【必须】修改)

In [5]:
DATASET_TYPE = 'CustomDataset'  # 必须和刚才注册的类名一致
DATA_ROOT = 'data/Watermelon87_Semantic_Seg_Mask' # 数据集存放的根目录

### 🛠️ 区域二：超参数调优区 (训练时根据显卡和需求【自由】修改)

In [6]:
BATCH_SIZE = 4           # 每次喂给显卡几张图？(显存大可以调大，如 8, 16。报错 OOM 就调小到 2)
NUM_WORKERS = 8          # 用几个 CPU 核心来搬运数据？(通常设为 4 或 8)
CROP_SIZE = (512, 512)   # 图像在训练时会被裁剪成多大？(决定了模型感受野和显存占用)
RESIZE_SCALE = (512, 512)# 图像在验证/测试时被缩放到的尺寸

### ⚙️ 区域三：底层构建区 (通常【不需要】修改，除非你想做更细致的数据增强)

**【1】训练数据流水线 (Train Pipeline)**

作用：定义一张原始图片从硬盘读取出来，到最终送入显卡计算前，要经历哪些“步骤”(数据增强)。

目的：通过随机变化，让模型见识更多样的数据，防止过拟合，提升泛化能力。

In [7]:
train_pipeline = [
    # 第1步：从硬盘读取原始彩色图像（默认读取为 BGR 格式的 3 通道矩阵）
    dict(type='LoadImageFromFile'),

    # 第2步：从硬盘读取对应的语义分割标注图（单通道的灰度图，里面每个像素值代表一个类别ID）
    dict(type='LoadAnnotations'),

    # 第3步：随机缩放 (Multi-scale Training)
    # 逻辑：每次读取图片时，按照 ratio_range (0.5到2.0倍) 随机生成一个缩放比例，应用在 RESIZE_SCALE 上。
    # 好处：让模型既能认出近处（大尺寸）的西瓜，也能认出远处（小尺寸）的西瓜。keep_ratio=True 保证长宽比不变，西瓜不会被压扁。
    dict(type='RandomResize', scale=RESIZE_SCALE, ratio_range=(0.5, 2.0), keep_ratio=True), 

    # 第4步：随机裁剪
    # 逻辑：由于显卡内存有限，且卷积神经网络通常需要固定尺寸的输入，所以从大图里随机抠出一块 crop_size 大小的区域。
    # 细节：cat_max_ratio=0.75 是一个极其聪明的设定！如果抠出来的这块区域里，某一个单一类别（比如全是大面积的背景）占据了 75% 以上，系统会放弃这块区域重新抠图。这保证了模型每次学习都能看到包含丰富边界和多类别的有效信息。
    dict(type='RandomCrop', crop_size=CROP_SIZE, cat_max_ratio=0.75),
                   
    # 第5步：随机水平翻转
    # 逻辑：以 50% 的概率把图片像照镜子一样左右翻转。（注意：原图翻转时，Mask标注图也会严丝合缝地跟着翻转！）
    dict(type='RandomFlip', prob=0.5),

    # 第6步：光学畸变 (Color Jitter)
    # 逻辑：随机改变图像的亮度、对比度、饱和度、色相。
    # 好处：模拟不同天气、不同光照条件、不同摄像机传感器拍出的照片，让模型学会只认西瓜的纹理，而不是死记硬背特定的光线。                                                 
    dict(type='PhotoMetricDistortion'),

    # 第7步：打包输入数据
    # 逻辑：把上述处理好的 Numpy 矩阵统一转换为 PyTorch 的 Tensor 格式，并打包成 MMSegmentation 底层代码能够认识的字典结构。                                                  
    dict(type='PackSegInputs')
]

**【2】验证/测试数据流水线 (Test Pipeline)**

作用：模型考试时的标准流程。

注意：考试必须公平，所以绝对不能有“随机翻转”、“随机裁剪”等数据增强操作

In [8]:
test_pipeline = [
    dict(type='LoadImageFromFile'),
    # 仅做固定尺寸的缩放，不加入随机范围
    dict(type='Resize', scale=RESIZE_SCALE, keep_ratio=True),
    # 读取标注图用于计算考试分数（准确率）
    dict(type='LoadAnnotations'),
    dict(type='PackSegInputs')
]

**【3】数据加载器 (DataLoader)**

作用：把 Pipeline 处理好的单张图片，打包成批次 (Batch) 喂给模型。

In [9]:
train_dataloader = dict(
    batch_size=BATCH_SIZE,   # 每次抓取几张图作为一个批次
    num_workers=NUM_WORKERS, # 雇佣几个 CPU 线程去后台并行搬运数据（防止显卡等 CPU 读图）
    persistent_workers=True, # 训练完一个 Epoch 后，不停止 CPU ，保持进程存活，大幅加快下一个 Epoch 的启动速度
    sampler=dict(type='DefaultSampler', shuffle=True), # 训练时必须开启打乱 (shuffle=True)，保证模型每次学到的顺序都不一样
    dataset=dict(
        type=DATASET_TYPE,
        data_root=DATA_ROOT,
        data_prefix=dict(img_path='img_dir/train', seg_map_path='ann_dir/train'), # 明确指出训练集的子文件夹
        pipeline=train_pipeline)) # 绑定上面写好的训练流水线

val_dataloader = dict(
    batch_size=1, # ⚠️ 核心规则：验证和测试时，为了评估指标计算绝对精准（避免多图 padding 造成的边缘误差），通常强制设定每次只处理 1 张图。
    num_workers=NUM_WORKERS,
    persistent_workers=True,
    sampler=dict(type='DefaultSampler', shuffle=False), # 考试不需要打乱顺序
    dataset=dict(
        type=DATASET_TYPE,
        data_root=DATA_ROOT,
        data_prefix=dict(img_path='img_dir/val', seg_map_path='ann_dir/val'),
        pipeline=test_pipeline))

**【4】配置打包与导出**

In [10]:
cfg = ConfigDict(
    train_pipeline=train_pipeline,
    test_pipeline=test_pipeline,
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    test_dataloader=val_dataloader, # 测试过程和验证过程使用同一套加载逻辑
    
    # 评估器配置：考试的评分标准是什么？
    # 'IoUMetric' 是专门用于语义分割的评估模块，'mIoU' (Mean Intersection over Union，平均交并比) 是业界最公认的分割好坏评价指标。
    val_evaluator=dict(type='IoUMetric', iou_metrics=['mIoU']),
    test_evaluator=dict(type='IoUMetric', iou_metrics=['mIoU'])
)

# 5.保存文件

In [11]:

# 保存文件
save_path = os.path.join(CONFIG_DIR, 'CustomDataset_pipeline.py')
with open(save_path, 'w') as f:
    f.write(f"# 自定义数据集 Pipeline 配置\n")
    for key, value in cfg.items():
        f.write(f"{key} = {repr(value)}\n\n")

print(f"🎉 核心配置文件已成功生成: {save_path}")

🎉 核心配置文件已成功生成: My-Configs/CustomDataset_pipeline.py
